<a href="https://colab.research.google.com/github/mattany/thesis/blob/main/RAG/Connect_to_vector_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### imports

In [1]:
!pip install llama-index
!pip install llama-index-embeddings-huggingface
!pip install peft
!pip install auto-gptq
!pip install optimum
!pip install bitsandbytes
!pip install llama-index-vector-stores-chroma
!pip install chromadb
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.3 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvi

In [2]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.schema import Document
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
import pandas as pd
import re
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


### Define Settings

In [3]:
# import any embedding model on HF hub (https://huggingface.co/spaces/mteb/leaderboard)
# Settings.embed_model = HuggingFaceEmbedding(model_name="dunzhang/stella_en_1.5B_v5")
# Settings.embed_model = HuggingFaceEmbedding(model_name="thenlper/gte-large") # alternative model

Settings.llm = None
Settings.chunk_size = 256
Settings.chunk_overlap = 25

LLM is explicitly disabled. Using MockLLM.


### mount google drive

In [5]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


### Load vector store index

In [6]:
db = chromadb.PersistentClient(path="/content/drive/MyDrive/RAG/content/chroma_db")
chroma_collection = db.get_or_create_collection("solar-system")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
index = VectorStoreIndex.from_vector_store(
    vector_store,
    embed_model=HuggingFaceEmbedding(model_name="dunzhang/stella_en_1.5B_v5"),
)

modules.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/174k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

2_Dense_1024/config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

### Set Up Search Function

In [7]:
# set number of docs to retreive
top_k = 5

# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=top_k,
)

In [39]:
# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.3)],
)

### Retrieve Relevant Docs

In [46]:
# query documents
query = "the temperature of the upper part of the Martian atmosphere"
response = query_engine.query(query)

In [47]:
# response
# reformat response
context = "Context:\n"
for i, node in enumerate(response.source_nodes):
    print(f"Document {i+1}:")
    print(node.score)
    print(node.metadata)
    print(node.text)

    context = context + node.text + "\n\n"

print(context)

Document 1:
0.411115595865061
{'id': '78566', 'url': 'https://en.wikipedia.org/wiki/Horse%20latitudes', 'title': 'Horse latitudes'}
If the low-level relative humidity rises towards 100 percent overnight, fog can form.
Document 2:
0.4046960053841153
{'id': '1096809', 'url': 'https://en.wikipedia.org/wiki/Polar%20night', 'title': 'Polar night'}
It occurs at latitudes between 67°24’ and 72°34’ North or South, when the Sun does not rise, only civil twilight visible.
Document 3:
0.4041760333139158
{'id': '18161217', 'url': 'https://en.wikipedia.org/wiki/US-KMO', 'title': 'US-KMO'}
Satellites
Document 4:
0.400337397872234
{'id': '2344032', 'url': 'https://en.wikipedia.org/wiki/Cybele%20asteroids', 'title': 'Cybele asteroids'}
417955 Mallama
Document 5:
0.39910820213162174
{'id': '56323438', 'url': 'https://en.wikipedia.org/wiki/ICEYE', 'title': 'ICEYE'}
ICEYE uses commercially available off-the-shelf components as much as possible, despite increased risk of hardware failure.
Context:
If the 

In [11]:
# from huggingface_hub import notebook_login

# notebook_login()

### Import LLM

In [12]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)


==((====))==  Unsloth 2024.8: Fast Llama patching. Transformers = 4.43.3.
   \\   /|    GPU: NVIDIA L4. Max memory: 22.168 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.1+cu121. CUDA = 8.9. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.26.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

### Use LLM

### No context zero shot prompt

In [31]:
alpaca_prompt = """Below is a scientific question, paired with context that you can use to answer the question. Write a response that appropriately completes the request.

### Question:
{}

### Context:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        query, # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
outputs = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

# outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
res = tokenizer.batch_decode(outputs)


<|begin_of_text|>Below is a scientific question, paired with context that you can use to answer the question. Write a response that appropriately completes the request.

### Question:
what are the names of the first people who landed on the moon?

### Context:


### Response:
The first people who landed on the moon were Neil Armstrong and Buzz Aldrin. They landed on July 20, 1969, and were the first humans to walk on the moon's surface. Armstrong was the first person to step on the moon, and he said, "That's one small step for man, one giant leap for mankind."<|end_of_text|>


In [32]:
alpaca_prompt = """Below is a scientific question, paired with context that you can use to answer the question. Write a response that appropriately completes the request.

### Question:
{}

### Context:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        query, # instruction
        context, # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
outputs = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

# outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
res = tokenizer.batch_decode(outputs)


<|begin_of_text|>Below is a scientific question, paired with context that you can use to answer the question. Write a response that appropriately completes the request.

### Question:
what are the names of the first people who landed on the moon?

### Context:
Context:
Moon landing

A Moon landing or lunar landing is the arrival of a spacecraft on the surface of the Moon. This includes both crewed and robotic missions. The first human-made object to touch the Moon was the Soviet Union's Luna 2, on 13 September 1959.

The United States' Apollo 11 was the first crewed mission to land on the Moon, on 20 July 1969. There were six crewed U.S. landings between 1969 and 1972, and numerous uncrewed landings, with no soft landings happening between 22 August 1976 and 14 December 2013.

The United States is the only country to have successfully conducted crewed missions to the Moon, with the last departing the lunar surface in December 1972. All soft landings took place on the near side of the M

In [25]:
print(res[0].split("Response:\n")[1])


Sunrise equation
The sunrise equation or sunset equation can be used to derive the time of sunrise or sunset for any solar declination and latitude in terms of local solar time when sunrise and sunset actually occur.

Formulation
It is formulated as:

where:
 is the solar hour angle at either sunrise (when negative value is taken) or sunset (when positive value is taken);
 is the latitude of the observer on the Earth;
 is the sun declination.

Principles 
The Earth rotates at an angular velocity of 15°/hour. Therefore, the expression, where  is in degree, gives the interval of time in hours from sunrise to local solar noon or from local solar noon to sunset.

The sign convention is typically that the observer latitude  is 0 at the equator, positive for the Northern Hemisphere and negative for the Southern Hemisphere, and the solar declination  is 0 at the vernal and autumnal equinoxes when the sun is exactly above the equator, positive during the Northern Hemisphere summer and negative

In [14]:
# prompt = "what equation can be used to derive the time of sunrise or sunset?"

In [ ]:
response = generate(model, tokenizer, prompt="hello", verbose=True)

In [ ]:
model.eval()

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(input_ids=inputs["input_ids"].to("cuda"), max_new_tokens=280)

print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# prompt (with context)
prompt_template_w_context = lambda context, comment: f"""[INST]ShawGPT, functioning as a virtual data science consultant on YouTube, communicates in clear, accessible language, escalating to technical depth upon request. \
It reacts to feedback aptly and ends responses with its signature '–ShawGPT'. \
ShawGPT will tailor the length of its responses to match the viewer's comment, providing concise acknowledgments to brief expressions of gratitude or feedback, \
thus keeping the interaction natural and engaging.

{context}
Please respond to the following comment. Use the context above if it is helpful.

{comment}
[/INST]
"""

In [ ]:
prompt = prompt_template_w_context(context, comment)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(input_ids=inputs["input_ids"].to("cuda"), max_new_tokens=280)

print(tokenizer.batch_decode(outputs)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST]ShawGPT, functioning as a virtual data science consultant on YouTube, communicates in clear, accessible language, escalating to technical depth upon request. It reacts to feedback aptly and ends responses with its signature '–ShawGPT'. ShawGPT will tailor the length of its responses to match the viewer's comment, providing concise acknowledgments to brief expressions of gratitude or feedback, thus keeping the interaction natural and engaging.

Context:
Some of the controversy might be explained by the observation that log-
normal distributions behave like Gaussian for low sigma and like Power Law
at high sigma [2].
However, to avoid controversy, we can depart (for now) from whether some
given data fits a Power Law or not and focus instead on fat tails.
Fat-tailedness — measuring the space between Mediocristan
and Extremistan
Fat Tails are a more general idea than Pareto and Power Law distributions.
One way we can think about it is that “fat-tailedness” is the degree to which
